# Resolution step: YOLO11s 4-class at imgsz 1280 rect (1280 x 480) instead of 1024 (1024 x 384)
Source strips are 2800 x 1024; training at 1024 keeps 37 % of the native width, 1280 keeps 46 %.
Same data as notebook `yolo11s-wood-4cls-balanced` v1 (build_balanced.py, seed 42, rebuilt here; preview names are
checked against v1 to confirm an identical build). Same config: yolo11s.pt, rect, batch 32, lr0 0.001, patience 40, seed 42.

Runs: sqrt-balanced @1280, plain @1280.

**Pre-registered rule (fixed before any result):** the final 4-class model is the one with the highest 4-class *val* mAP50 among
sqrt@1024 (75.3), plain@1024 (73.1), sqrt@1280, plain@1280. Both 1280 models are then evaluated on test exactly once
(at imgsz 1280). No decision uses test numbers.

In [ ]:
!pip -q install ultralytics==8.4.142
!nvidia-smi --query-gpu=name,memory.total --format=csv
import ultralytics, torch, os, glob, shutil, time, json, subprocess, sys
T0 = time.time()
print(ultralytics.__version__, torch.__version__, torch.cuda.device_count(), 'GPUs')
yml = glob.glob('/kaggle/input/**/dataset_clean/data.yaml', recursive=True)
assert yml, os.listdir('/kaggle/input')
SRC = os.path.dirname(yml[0])
hits = glob.glob('/kaggle/input/**/build_balanced.py', recursive=True)
assert hits, 'build_balanced.py not found - is wood-defects-scripts attached?'
SCRIPTS = os.path.dirname(hits[0]); print('scripts dir:', SCRIPTS, os.listdir(SCRIPTS))
DATA = '/kaggle/tmp/dataset_clean'
if not os.path.isdir(DATA + '/train/images'):
    t = time.time(); shutil.copytree(SRC, DATA); print(f'copied dataset to local disk in {time.time()-t:.0f}s')

In [ ]:
OUT = '/kaggle/tmp/dataset_4cls'
t = time.time()
rc = subprocess.call([sys.executable, f'{SCRIPTS}/build_balanced.py', '--src', DATA, '--out', OUT, '--seed', '42'])
assert rc == 0, 'build_balanced.py failed'
shutil.copytree(f'{OUT}/previews', '/kaggle/working/previews', dirs_exist_ok=True)
shutil.copy2(f'{OUT}/balance_report.json', '/kaggle/working/balance_report.json')
V1 = ['train_sqrt_042fff_104900047s0216.jpg', 'train_sqrt_24b8d1_111200005s0176.jpg', 'train_sqrt_5416f5_111200047s0174.jpg',
      'train_sqrt_5afb6d_106100037s0357.jpg', 'train_sqrt_c826f2_109300034s0179.jpg', 'train_sqrt_e72b6c_100500037s0412.jpg']
now = sorted(f for f in os.listdir(f'{OUT}/previews') if f.startswith('train_sqrt_'))
SAME = now == V1
print('identical build to v1 (preview names match):', SAME, now)
print(open('/kaggle/working/balance_report.json').read()[:600])
print(f'built in {time.time()-t:.0f}s')

In [ ]:
def train(v, imgsz, batch, name):
    cmd = [sys.executable, f'{SCRIPTS}/train_yolo11m.py', '--data', f'{OUT}/data_{v}.yaml', '--name', name,
           '--device', '0', '--model', 'yolo11s.pt', '--batch', str(batch), '--lr0', '0.001', '--patience', '40',
           '--imgsz', str(imgsz), '--project', '/kaggle/working/outputs', '--persist', '/kaggle/working/persist']
    print(' '.join(cmd)); t = time.time(); rc = subprocess.call(cmd)
    ok = os.path.exists(f'/kaggle/working/outputs/{name}/weights/best.pt')
    print(name, 'exit', rc, 'best.pt OK' if ok else 'NO best.pt', f'{(time.time()-t)/3600:.2f} h')
    return ok, round((time.time() - t) / 3600, 2)

RUNS = [('sqrt', 1280, 'yolo11s_4cls_sqrt_1280'), ('plain', 1280, 'yolo11s_4cls_plain_1280')]
done, hours = {}, {}
for v, imgsz, name in RUNS:
    if (time.time() - T0) / 3600 > 8.0:
        print('SKIP', name); continue
    ok, h = train(v, imgsz, 32, name)
    if not ok:                                   # e.g. CUDA OOM in the first epoch: retry once at batch 16
        name = name + '_b16'; ok, h2 = train(v, imgsz, 16, name); h += h2
    hours[name] = h
    if ok:
        done[name] = (v, imgsz)
        print(open(f'/kaggle/working/outputs/{name}/results.csv').read()[-600:])
json.dump(dict(done=done, hours=hours, identical_build_to_v1=SAME), open('/kaggle/working/runs_done.json', 'w'), indent=2)
print(done, hours)

In [ ]:
val = {'yolo11s_4cls_sqrt (1024, notebook v1)': 75.3, 'yolo11s_4cls_plain (1024, notebook v1)': 73.1}
for name, (v, imgsz) in done.items():
    out = f'/kaggle/working/eval/val_{name}'
    subprocess.call([sys.executable, f'{SCRIPTS}/evaluate.py', '--weights', f'/kaggle/working/outputs/{name}/weights/best.pt',
                     '--data', f'{OUT}/data_{v}.yaml', '--split', 'val', '--imgsz', str(imgsz), '--device', '0', '--out', out])
    md = open(f'{out}/val_metrics.md').read(); print(name); print(md)
    val[name] = float([l.split('|') for l in md.splitlines() if l.startswith('| ALL')][0][5])
FINAL = max(val, key=val.get)
json.dump(dict(val_map50=val, final=FINAL, rule='highest 4-class val mAP50 among sqrt/plain at 1024 and 1280'),
          open('/kaggle/working/selection.json', 'w'), indent=2)
print('val mAP50:', val, '-> final 4-class model by the pre-registered rule:', FINAL)

## Test evaluation: exactly once per 1280 model. No decision is made from these numbers.

In [ ]:
for name, (v, imgsz) in done.items():
    out = f'/kaggle/working/eval/test_{name}'
    subprocess.call([sys.executable, f'{SCRIPTS}/evaluate.py', '--weights', f'/kaggle/working/outputs/{name}/weights/best.pt',
                     '--data', f'{OUT}/data_{v}.yaml', '--split', 'test', '--imgsz', str(imgsz), '--device', '0', '--out', out])
    print(name); print(open(f'{out}/test_metrics.md').read())
print(f'total wall-clock {(time.time() - T0) / 3600:.2f} h')